# `tournament` -- 2026 Points-Table Simulation

**Ported from `project_gagan` notebook cells 60-64.** Builds two IPL 2026 points tables -- one from what **actually happened**, one from what the pre-match model **predicted would happen** -- and compares them. This is one of project_gagan's unique pieces of functionality with no equivalent in project_hrishav: it tests the pre-match model's predictions in a way a plain accuracy score can't, by asking "if we'd trusted this model's picks for the whole season, would the resulting standings have looked anything like reality?"

Standard T20 league scoring throughout: 2 points for a win, 0 for a loss, ranked by points then by Net Run Rate (NRR) as the tiebreaker.

In [1]:
import pandas as pd

from src.data import ABBREV

## `actual_points_table()` -- what really happened in 2026

Iterates through every 2026 pre-match record (`pm26`), crediting 2 points to whichever team actually won. In parallel, it accumulates the four running totals needed for **Net Run Rate**: runs scored/balls faced when batting (`nrr_rf`/`nrr_bf`), and runs conceded/balls bowled when bowling (`nrr_ra`/`nrr_ba`), pulled from the actual ball-by-ball 2026 data (`ig26`) for each match.

**The all-out adjustment**: if a team is bowled out (10 wickets down) before facing all 120 balls, cricket's NRR convention treats them as having faced the **full quota** of balls anyway (`b1e = 120 if w1 >= 10 else b1`) -- otherwise a team that's skittled out cheaply in very few balls would get an unfairly inflated run rate from the small denominator. This is standard cricket scoring convention, not a project-specific choice.

The private `nrr()` closure computes the standard formula: *(run rate scored) - (run rate conceded)*. The final table is sorted by points, then NRR, both descending -- exactly how a real IPL points table is ordered -- and 1-indexed (`index += 1`) so it displays as "1st place, 2nd place, ..." rather than 0-indexed.

In [2]:
def actual_points_table(pm26: pd.DataFrame, ig26: pd.DataFrame) -> pd.DataFrame:
    """Build the actual points table (with Net Run Rate) from real 2026 results."""
    teams_2026 = list(ABBREV.values())
    pts = {t: 0 for t in teams_2026}
    wins = {t: 0 for t in teams_2026}
    losses = {t: 0 for t in teams_2026}
    nrr_rf = {t: 0.0 for t in teams_2026}
    nrr_bf = {t: 0 for t in teams_2026}
    nrr_ra = {t: 0.0 for t in teams_2026}
    nrr_ba = {t: 0 for t in teams_2026}

    for _, r in pm26.iterrows():
        t1, t2, winner = r["team_bat_first"], r["team_bowl_first"], r["actual_winner"]
        loser = t2 if winner == t1 else t1
        pts[winner] += 2
        wins[winner] += 1
        losses[loser] += 1
        mid = r["match_id"]
        m1 = ig26[(ig26["match_id"] == mid) & (ig26["innings"] == 1)].sort_values("team_balls")
        m2 = ig26[(ig26["match_id"] == mid) & (ig26["innings"] == 2)].sort_values("team_balls")
        if len(m1) == 0 or len(m2) == 0:
            continue
        l1, l2 = m1.iloc[-1], m2.iloc[-1]
        s1, b1, w1 = l1["team_runs"], int(l1["team_balls"]), int(l1["team_wicket"])
        s2, b2, w2 = l2["team_runs"], int(l2["team_balls"]), int(l2["team_wicket"])
        b1e = 120 if w1 >= 10 else b1
        b2e = 120 if w2 >= 10 else b2
        nrr_rf[t1] += s1; nrr_bf[t1] += b1e; nrr_ra[t1] += s2; nrr_ba[t1] += b2e
        nrr_rf[t2] += s2; nrr_bf[t2] += b2e; nrr_ra[t2] += s1; nrr_ba[t2] += b1e

    def nrr(t):
        f = nrr_rf[t] / nrr_bf[t] * 6 if nrr_bf[t] > 0 else 0
        a = nrr_ra[t] / nrr_ba[t] * 6 if nrr_ba[t] > 0 else 0
        return f - a

    table_df = pd.DataFrame(
        [{"Team": t, "P": wins[t] + losses[t], "W": wins[t], "L": losses[t],
          "Pts": pts[t], "NRR": round(nrr(t), 3)} for t in teams_2026]
    )
    table_df = table_df.sort_values(["Pts", "NRR"], ascending=False).reset_index(drop=True)
    table_df.index += 1
    return table_df

## `predicted_points_table()` -- what the pre-match model would have predicted

Same 2-points-per-win scoring, but this time using the pre-match model's own predictions (`pre_df`, produced by `pipeline.py::evaluate_2026_pre_match`) instead of the real results. **No NRR here** -- NRR depends on the margin of victory/defeat, which a simple win/loss classifier has no opinion about, so this table is ranked by predicted points alone.

In [3]:
def predicted_points_table(pre_df: pd.DataFrame) -> pd.DataFrame:
    """Build the predicted points table from the pre-match model's picks."""
    teams_2026 = list(ABBREV.values())
    pts_p = {t: 0 for t in teams_2026}
    wins_p = {t: 0 for t in teams_2026}
    losses_p = {t: 0 for t in teams_2026}
    for _, r in pre_df.iterrows():
        t1, t2 = r["bat_first"], r["bowl_first"]
        w = r["pred"]
        l = t2 if w == t1 else t1
        pts_p[w] += 2
        wins_p[w] += 1
        losses_p[l] += 1

    pred_table = pd.DataFrame(
        [{"Team": t, "P": wins_p[t] + losses_p[t], "W_pred": wins_p[t],
          "L_pred": losses_p[t], "Pts_pred": pts_p[t]} for t in teams_2026]
    ).sort_values("Pts_pred", ascending=False).reset_index(drop=True)
    pred_table.index += 1
    return pred_table

## `compare_tables()` -- how close did the prediction get?

Two summary statistics that are easier to reason about than the full tables side by side:
- **Table topper**: did the model correctly predict which team   would finish 1st? (Verified result: yes -- Royal Challengers   Bengaluru, correctly predicted.)
- **Top-4 overlap**: of the actual top 4 (the playoff   qualifiers), how many did the model also place in its   predicted top 4? (Verified result: 3 out of 4.)

Set intersection (`top4_actual & top4_pred`) is the natural tool here since team order *within* the top 4 doesn't matter for this comparison -- only membership does.

In [4]:
def compare_tables(table_df: pd.DataFrame, pred_table: pd.DataFrame) -> dict:
    """Compare actual vs. predicted tournament outcomes: table topper + top-4 overlap."""
    top4_actual = set(table_df.head(4)["Team"])
    top4_pred = set(pred_table.head(4)["Team"])
    overlap = top4_actual & top4_pred
    return {
        "actual_topper": table_df.iloc[0]["Team"],
        "predicted_topper": pred_table.iloc[0]["Team"],
        "topper_correct": table_df.iloc[0]["Team"] == pred_table.iloc[0]["Team"],
        "top4_actual": sorted(top4_actual),
        "top4_predicted": sorted(top4_pred),
        "top4_overlap": sorted(overlap),
        "top4_overlap_count": len(overlap),
    }